# Обучение классических классификаторов

Используем подготовленные данные из `02_preparation.ipynb`. Сравним классические модели для двух целей: CLASS — паттерн КТГ, NSP — оценка состояния. Для каждой цели обучается отдельный классификатор. Эксперименты сохраняются в MLflow.

Train используется для обучения, validation — для сравнения и выбора настроек. Test сохраняется для итоговой оценки после экспериментов с MLP и фиксации выбранных моделей.

## 1. Загрузка подготовленных данных

Загрузим train.csv, validation.csv и test.csv из `runtime/data/prepared/`. Отделим 21 входной признак от CLASS, NSP и служебных FileName/SegFile. Проверим одинаковый состав колонок и отсутствие пересечения исследований между частями.

Результат: входы и две цели для каждой части выборки. Test в расчётах качества на этом этапе не используется.

## 2. Метрики и настройка MLflow

Основная метрика — macro-F1: каждый класс учитывается с одинаковым весом. Дополнительно рассчитаем accuracy, precision и recall по классам. Для Pathologic отдельно рассмотрим recall и precision.

В MLflow будем сохранять цель, тип модели, параметры, seed, метрики train/validation, отчёт классификации и обученный pipeline.

Результат: единый способ оценки и регистрации экспериментов, позволяющий сравнивать модели отдельно для CLASS и NSP.

## 3. Базовый ориентир: DummyClassifier

Для обеих целей обучим DummyClassifier, который всегда выбирает самый частый класс train. Оценим его на validation по тем же метрикам, что и остальные модели.

Результат: исходный уровень качества, с которым сравниваются обучаемые классификаторы.

## 4. Обучение логистической регрессии и случайного леса

Обучим LogisticRegression и RandomForestClassifier отдельно для CLASS и NSP. Для логистической регрессии включим StandardScaler в sklearn Pipeline. Scaler обучается только на train; validation преобразуется уже обученным scaler. Для случайного леса масштабирование не требуется.

Результат: четыре обученные модели и зарегистрированные в MLflow эксперименты с начальными настройками.

## 5. Сравнение моделей на validation

Составим отдельные таблицы метрик для CLASS и NSP. Рассмотрим матрицы ошибок и precision/recall по классам: какие категории путаются и насколько хорошо обнаруживается Pathologic. Сравним качество train и validation, чтобы заметить признаки переобучения.

Результат: обоснованный выбор моделей для подбора параметров и краткий вывод об их сильных и слабых сторонах.

## 6. Подбор параметров

Зададим небольшой набор настроек выбранных моделей. Каждый вариант обучим на train и оценим на validation; результаты сохраним в MLflow. Число экспериментов ограничим, чтобы сравнение оставалось понятным.

Результат: выбранные настройки для каждой цели по macro-F1 на validation с учётом метрик отдельных классов. Test не участвует в подборе.

## 7. Выводы и сохранение выбранных моделей

Зафиксируем выбранный классический классификатор для CLASS и для NSP, его параметры и качество на validation. Сохраним обученные модели вместе с необходимыми преобразованиями, порядком входных признаков и соответствием кодов целевых классов.

Результат: воспроизводимые классические модели для дальнейшего сравнения с MLP. Итоговая оценка на test выполняется после завершения сравнения и выбора моделей.